# 6.5 Transformer Text Classification

這份 Notebook 使用 Keras layer 建立小型 Transformer encoder 文字分類模型。


## 1. 載入套件


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

tf.keras.utils.set_random_seed(42)


## 2. 建立客服意圖分類資料


In [ ]:
CLASS_NAMES = np.array(['billing', 'shipping', 'technical'])

def make_intent_data(repeats=90, seed=42):
    rng = np.random.default_rng(seed)
    templates = {
        'billing': [
            'my invoice has a wrong charge',
            'please refund the payment',
            'the billing amount is incorrect',
            'i need a receipt for my order',
            'my credit card was charged twice',
        ],
        'shipping': [
            'my package arrived late',
            'where is the delivery tracking number',
            'the shipment is delayed today',
            'please change my delivery address',
            'the courier lost my package',
        ],
        'technical': [
            'i cannot login to the app',
            'the website shows an error message',
            'the feature crashes after update',
            'my password reset link failed',
            'the dashboard does not load',
        ],
    }
    prefixes = ['hello', 'urgent', 'please help', 'support team', 'hi']
    texts, labels = [], []
    for label, class_name in enumerate(CLASS_NAMES):
        for _ in range(repeats):
            for sentence in templates[class_name]:
                texts.append(rng.choice(prefixes) + ' ' + sentence)
                labels.append(label)
    texts = np.array(texts, dtype=object)
    labels = np.array(labels, dtype='int64')
    idx = rng.permutation(len(labels))
    return texts[idx], labels[idx]

def split_texts(texts, labels):
    x_train, x_temp, y_train, y_temp = train_test_split(
        texts, labels, test_size=0.3, random_state=42, stratify=labels
    )
    x_valid, x_test, y_valid, y_test = train_test_split(
        x_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
    )
    return x_train, x_valid, x_test, y_train, y_valid, y_test

texts, labels = make_intent_data(repeats=90, seed=3)
x_train, x_valid, x_test, y_train, y_valid, y_test = split_texts(texts, labels)
print(len(x_train), len(x_valid), len(x_test))
pd.DataFrame({'text': x_train[:6], 'label': CLASS_NAMES[y_train[:6]]})


## 3. 建立 TextVectorization


In [ ]:
MAX_TOKENS = 1500
SEQ_LEN = 14
vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode='int',
    output_sequence_length=SEQ_LEN,
)
vectorizer.adapt(x_train)
print(vectorizer.get_vocabulary()[:25])


## 4. 建立 Token + Position Embedding 與 Transformer Block


In [ ]:
class TokenAndPositionEmbedding(tf.keras.layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = tf.keras.layers.Embedding(input_dim=maxlen, output_dim=embed_dim)
        self.maxlen = maxlen

    def call(self, x):
        positions = tf.range(start=0, limit=self.maxlen, delta=1)
        return self.token_emb(x) + self.pos_emb(positions)

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(ff_dim, activation='relu'),
            tf.keras.layers.Dense(embed_dim),
        ])
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(rate)
        self.dropout2 = tf.keras.layers.Dropout(rate)

    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)


## 5. 建立 Transformer 文字分類模型


In [ ]:
EMBED_DIM = 32
NUM_HEADS = 2
FF_DIM = 64

inputs = tf.keras.Input(shape=(), dtype=tf.string)
x = vectorizer(inputs)
x = TokenAndPositionEmbedding(SEQ_LEN, MAX_TOKENS, EMBED_DIM)(x)
x = TransformerBlock(EMBED_DIM, NUM_HEADS, FF_DIM)(x)
x = tf.keras.layers.GlobalAveragePooling1D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


## 6. 訓練與評估


In [ ]:
history = model.fit(x_train, y_train, validation_data=(x_valid, y_valid), epochs=12, batch_size=32, verbose=0)
pd.DataFrame(history.history)[['accuracy', 'val_accuracy']].plot(figsize=(8, 4), title='Transformer text accuracy')
plt.ylim(0, 1.05)
plt.show()
y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
print(model.evaluate(x_test, y_test, verbose=0, return_dict=True))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))


## 7. 預測新客服訊息


In [ ]:
new_texts = np.array(['refund the card charge please', 'the courier delayed my shipment', 'the dashboard crashes'], dtype=object)
prob = model.predict(new_texts, verbose=0)
pd.DataFrame(prob, columns=CLASS_NAMES, index=new_texts)


## 8. 如何套用自己的資料

短句分類可以先使用小型 Transformer encoder 理解結構。若資料量不大，請同時和 LSTM、CNN baseline 比較，並注意過擬合。


## 9. 小結

Transformer encoder 結合 token embedding、position embedding 與 self-attention。這個小型範例適合理解架構，實務最佳效果通常會再延伸到預訓練模型。
